<a href="https://colab.research.google.com/github/samradnyi-AIMLtech/RAG-Based-Document-Intelligence-System/blob/main/final_of_Copy_of_RAG_Document_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install pypdf
!pip -q install sentence-transformers
!pip -q install faiss-cpu
!pip -q install transformers
!pip -q install accelerate
!pip -q install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 48.3 MB/s eta 0:00:00


In [2]:
import os
import re
import numpy as np
import faiss
import torch
import gradio as gr

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
# PDF files will be uploaded through the Gradio application.
# We initialize the document storage here.

all_pages = []
chunks = []

print("PDF document system initialized.")

PDF document system initialized.


In [4]:
def extract_text_from_pdf(pdf_path):

    reader = PdfReader(pdf_path)

    pages = []

    for page_number, page in enumerate(
        reader.pages,
        start=1
    ):

        text = page.extract_text()

        if text:

            pages.append({
                "document": os.path.basename(pdf_path),
                "page": page_number,
                "text": text
            })

    return pages


print("PDF extraction function ready.")

PDF extraction function ready.


In [5]:
def clean_text(text):

    text = re.sub(
        r'\s+',
        ' ',
        text
    )

    text = text.strip()

    return text


print("Text cleaning function ready.")

Text cleaning function ready.


In [7]:
def create_chunks(
    pages,
    chunk_size=800,
    overlap=300
):

    chunks = []

    for page in pages:

        text = page["text"]

        page_number = page["page"]

        document_name = page["document"]

        start = 0

        while start < len(text):

            end = start + chunk_size

            chunk_text = text[start:end]

            if chunk_text.strip():

                chunks.append({
                    "text": chunk_text,
                    "page": page_number,
                    "document": document_name
                })

            start += (
                chunk_size - overlap
            )

    return chunks


print("Chunking function ready.")

Chunking function ready.


In [8]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded!


In [9]:
#Embedding Helper

def create_embeddings(chunks):

    if not chunks:
        raise ValueError(
            "No chunks available. "
            "Upload and process PDFs first."
        )

    chunk_texts = [
        chunk["text"]
        for chunk in chunks
        if chunk["text"].strip()
    ]

    if not chunk_texts:
        raise ValueError(
            "The PDFs contain no extractable text."
        )

    embeddings = embedding_model.encode(
        chunk_texts,
        convert_to_numpy=True,
        show_progress_bar=False
    )

    embeddings = np.asarray(embeddings)

    if embeddings.ndim != 2:
        raise ValueError(
            f"Unexpected embedding shape: {embeddings.shape}"
        )

    return embeddings


print("Embedding function ready.")


Embedding function ready.


In [10]:
# FAISS

def create_faiss_index(embeddings):

    if embeddings is None:
        raise ValueError(
            "Embeddings have not been created."
        )

    if embeddings.ndim != 2:
        raise ValueError(
            f"Invalid embedding shape: {embeddings.shape}"
        )

    dimension = embeddings.shape[1]

    index = faiss.IndexFlatL2(
        dimension
    )

    index.add(
        embeddings.astype("float32")
    )

    return index


print("FAISS function ready.")

FAISS function ready.


In [11]:
def retrieve_documents(
    question,
    selected_documents=None,
    top_k=5
):

    question_embedding = (
        embedding_model.encode(
            [question],
            convert_to_numpy=True
        )
    )

    search_k = min(
        top_k * 5,
        index.ntotal
    )

    distances, indices = index.search(
        question_embedding.astype(
            "float32"
        ),
        search_k
    )

    results = []

    for distance, idx in zip(
        distances[0],
        indices[0]
    ):

        if idx >= len(chunks):
            continue

        chunk = chunks[idx]


       # Match base filenames cleanly across different system paths
        selected_filenames = [os.path.basename(doc) for doc in selected_documents] if selected_documents else []

        if (
            selected_filenames
            and chunk["document"] not in selected_filenames
        ):
            continue

        results.append({
            "text": chunk["text"],
            "page": chunk["page"],
            "document": chunk["document"],
            "distance": float(distance)
        })

        if len(results) >= top_k:
            break

    return results


print("Multi-document retrieval ready.")

Multi-document retrieval ready.


In [12]:
model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [13]:
def create_prompt(
    question,
    retrieved_documents,
    mode="Ask"
):

    context = ""
    for doc in retrieved_documents:
        context += f"Source ({doc['document']}, p.{doc['page']}): {doc['text']}\n\n"

    # Select brief task instruction based on mode
    if mode == "Summarize":
        instruction = "Summarize the key information from the context."
    elif mode == "Compare":
        instruction = "Compare similarities and differences in the context."
    elif mode == "Extract":
        instruction = "Extract key facts from the context in bullet points."
    elif mode == "Research":
        instruction = "Provide a detailed analysis using the context."
    else:
        instruction = "Answer the question directly using the context."

    prompt = f"""Task: {instruction} If the answer is not in the context, say "I could not find the answer in the uploaded documents."

Context:
{context}

Question: {question}
Answer:"""

    return prompt

In [14]:
def generate_answer(prompt):
    # Tokenize the prompt text directly
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(llm.device)

    with torch.no_grad():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Slice off the input prompt tokens, keeping ONLY the newly generated answer tokens
    input_len = inputs.input_ids.shape[1]
    generated_tokens = outputs[0][input_len:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer

In [15]:
def ask_documents(
    question,
    selected_documents=None,
    mode="Ask",
    top_k=5
):
    # Step 1: Retrieve relevant documents
    retrieved_docs = retrieve_documents(
        question,
        selected_documents,
        top_k
    )

    if not retrieved_docs:
        return (
            "I could not find any relevant information in the uploaded documents.",
            []
        )

    # Step 2: Create a prompt using the retrieved documents
    prompt = create_prompt(
        question,
        retrieved_docs,
        mode
    )

    # Step 3: Generate an answer using the LLM
    answer = generate_answer(
        prompt
    )

    return (
        answer,
        retrieved_docs
    )

In [16]:
import gradio as gr

In [17]:
all_pages = []
chunks = []
embeddings = None
index = None


In [18]:
#proccesing the pdf
def process_pdfs(files):

    global all_pages
    global chunks
    global embeddings
    global index

    if not files:

        return (
            "❌ Please upload at least one PDF.",
            gr.update(choices=[])
        )

    all_pages = []
    chunks = []
    embeddings = None
    index = None

    #here we extract the files

    for file in files:

        pdf_path = file

        pdf_pages = (
            extract_text_from_pdf(
                pdf_path
            )
        )

        all_pages.extend(
            pdf_pages
        )
         # Clean pages
    for page in all_pages:

        page["text"] = clean_text(
            page["text"]
        )

    # Create chunks
    chunks = create_chunks(
        all_pages,
        chunk_size=800,
        overlap=150
    )

    # Create embeddings
    chunk_texts = [
        chunk["text"]
        for chunk in chunks
    ]

    embeddings = (
        embedding_model.encode(
            chunk_texts,
            convert_to_numpy=True,
            show_progress_bar=False
        )
    )

    # Create FAISS index
    dimension = embeddings.shape[1]

    index = faiss.IndexFlatL2(
        dimension
    )

    index.add(
        embeddings.astype(
            "float32"
        )
    )

    documents = sorted(
        set(
            chunk["document"]
            for chunk in chunks
        )
    )

    status = f"""
  PDF processing completed

  Documents: {len(documents)}
  Pages: {len(all_pages)}
  Chunks: {len(chunks)}
  Vectors: {index.ntotal}
"""

    return (
        status,
        gr.update(
            choices=documents,
            value=documents
        )
    )

In [20]:
import gradio as gr

# ASK DOCUMENTS
def run_rag(
    question,
    selected_documents,
    mode
):

    if not question.strip():

        return (
            "⚠️ Please enter a question.",
            ""
        )

    if index is None:

        return (
            "⚠️ Please upload and process "
            "your PDFs first.",
            ""
        )

    # If nothing selected,
    # search all documents.
    if not selected_documents:

        selected_documents = sorted(
            set(
                chunk["document"]
                for chunk in chunks
            )
        )

    answer, retrieved = ask_documents(
    question=question,
    selected_documents=selected_documents,
    mode=mode,
    top_k=5
)

    source_text = ""

    for source in retrieved:

        source_text += (
            f"📄 {source['document']} "
            f"| Page {source['page']}\n"
        )

    return (
        answer,
        source_text
    )

In [ ]:
import gradio as gr

# APPLICATION UI
with gr.Blocks(
    title="RAG Document Intelligence"
) as demo:

    gr.Markdown(
        """
# RAG Document Intelligence System

### Multi-PDF • Multiple Intelligence Modes • Citations

Upload your PDF documents, process them,
select the documents you want to use,
and ask questions.
"""
    )

    with gr.Row():
        with gr.Column(
            scale=1
        ):

            gr.Markdown(
                "##   Upload Documents"
            )

            pdf_files = gr.File(
                label="Upload PDF files",
                file_count="multiple",
                file_types=[".pdf"],
                type="filepath"
            )

            process_button = gr.Button(
                "⚙️ Process / Check PDFs",
                variant="primary"
            )

            status = gr.Markdown(
                "No documents processed yet."
            )

            document_selector = (
                gr.CheckboxGroup(
                    label="Select PDFs",
                    choices=[],
                    value=[]
                )
            )
        with gr.Column(
            scale=2
        ):

            gr.Markdown(
                "##   Document Intelligence"
            )

            mode = gr.Radio(
                choices=[
                    "Ask",
                    "Summarize",
                    "Compare",
                    "Extract",
                    "Research"
                ],
                value="Ask",
                label="Choose Intelligence"
            )

            question = gr.Textbox(
                label="Ask your documents",
                placeholder=(
                    "Example: What are the "
                    "main technologies discussed?"
                ),
                lines=4
            )

            ask_button = gr.Button(
                "️ Ask",
                variant="primary"
            )

            gr.Markdown(
                "##   Answer"
            )

            answer = gr.Markdown()

            gr.Markdown(
                "##   Sources"
            )

            sources = gr.Textbox(
                lines=8,
                interactive=False
            )
            # PROCESS BUTTON
    process_button.click(
        fn=process_pdfs,
        inputs=pdf_files,
        outputs=[
            status,
            document_selector
        ]
    )
    #Ask button
    ask_button.click(
        fn=run_rag,
        inputs=[
            question,
            document_selector,
            mode
        ],
        outputs=[
            answer,
            sources
        ]
    )
    #Enter key
    question.submit(
        fn=run_rag,
        inputs=[
            question,
            document_selector,
            mode
        ],
        outputs=[
            answer,
            sources
        ]
    )


demo.launch(
    share=True,
    debug=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7ff3f5b7b28353e652.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
